In [15]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Your existing settings
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "practice_db"
DB_USER = "postgres"
DB_PASS = "mysecretpassword"

conn_str = f"postgresql://{DB_USER}:{quote_plus(DB_PASS)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_str)

def run_sql(query: str):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(query))
            df = pd.DataFrame(result.mappings())   # ← this handles SQLAlchemy 2.0+
            display(df)
    except Exception as e:
        print(f"Error executing query:\n{e}")

print("✅ Fixed SQL execution environment ready!")

✅ Fixed SQL execution environment ready!


In [24]:
sql="""
-- Ex 1. Above-average earners using a named CTE
-- Return: name, dept_name, salary, dept_avg, difference
-- Use CTE instead of correlated subquery or inline window
WITH calc AS (
SELECT e.name as name , d.dept_name as dept_name, e.salary as salary
,AVG(e.salary) Over (PARTITION BY dept_name) as dept_avg
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
ORDER BY dept_name, salary DESC
) 
SELECT 
    name,
    dept_name,
    salary, 
    ROUND (dept_avg, 2) as dept_avg,
    Round((salary - dept_avg ),0) as difference
FROM calc where 
salary >= dept_avg
ORDER BY dept_name, Salary desc
"""
run_sql(sql)



,dept_avg,dept_name,difference,name,salary
0,111250.00,Engineering,8750,Alice,120000.00
1,111250.00,Engineering,8750,Bob,120000.00
2,107333.33,Finance,7667,Henry,115000.00
3,107333.33,Finance,7667,Ivy,115000.00
4,99333.33,Marketing,5667,Eve,105000.00
5,99333.33,Marketing,5667,Frank,105000.00


In [25]:
sql = """
-- Ex 2. Simple multi-step pipeline: raw telemetry → daily average per server
-- CTEs: raw → daily_avg
-- Final: show server_id, date, avg_cpu for srv-01 only
-- 2. Simple raw → daily average pipeline
WITH daily AS (
    SELECT 
        server_id,
        collection_date,
        ROUND(AVG(cpu_utilization), 1) AS avg_cpu
    FROM telemetry
    GROUP BY server_id, collection_date
)
SELECT * 
FROM daily 
WHERE server_id = 'srv-01'
ORDER BY collection_date;
"""
run_sql(sql)

,avg_cpu,collection_date,server_id
0,45.7,2026-01-01,srv-01
1,79.6,2026-01-02,srv-01
2,91.4,2026-01-03,srv-01
3,79.2,2026-01-04,srv-01
4,40.0,2026-01-05,srv-01
5,14.1,2026-01-06,srv-01
6,4.6,2026-01-07,srv-01
7,9.6,2026-01-08,srv-01
8,46.9,2026-01-09,srv-01
9,86.1,2026-01-10,srv-01


In [27]:
sql = """
-- Ex 3. 4-step capacity-like pipeline
-- raw → daily_avg → 7d_rolling_max → status ('HIGH' if rolling_max > 85, else 'OK')
-- Return: server_id, date, rolling_max, status
WITH 
raw_filtered AS (
    SELECT * FROM telemetry
-- Add filtration if needed WHERE CLAUSE
),  
daily AS (
    SELECT 
        server_id,
        collection_date,
        ROUND(AVG(cpu_utilization), 1) AS avg_cpu
    FROM raw_filtered
    GROUP BY server_id, collection_date
), 
rolling AS(
    SELECT 
        *,
        ROUND(MAX(avg_cpu) OVER (
            PARTITION BY server_id 
            ORDER BY collection_date 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 1) AS roll_max_7d
    FROM daily
)SELECT 
    server_id,
    collection_date,
    roll_max_7d,
    CASE 
        WHEN roll_max_7d > 85 THEN 'HIGH'
        WHEN roll_max_7d > 70 THEN 'MEDIUM'
        ELSE 'OK'
    END AS status
FROM rolling
WHERE server_id = 'srv-03'
ORDER BY collection_date;


"""
run_sql(sql)

,collection_date,roll_max_7d,server_id,status
0,2026-01-01,44.6,srv-03,OK
1,2026-01-02,87.3,srv-03,HIGH
2,2026-01-03,93.1,srv-03,HIGH
3,2026-01-04,93.1,srv-03,HIGH
4,2026-01-05,93.1,srv-03,HIGH
5,2026-01-06,93.1,srv-03,HIGH
6,2026-01-07,93.1,srv-03,HIGH
7,2026-01-08,93.1,srv-03,HIGH
8,2026-01-09,93.1,srv-03,HIGH
9,2026-01-10,93.8,srv-03,HIGH
